## Configuration


In [ ]:
dbutils.widgets.text("catalog", "training_lab")
dbutils.widgets.text("schema", "student_01")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)
ns = f"{catalog}.{schema}"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
print("Working namespace:", ns)
from pyspark.sql import functions as F


## Imports and chronological split


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet
pdf = spark.table(ns+".day18_daily_demand").orderBy("ds").toPandas()
pdf["ds"] = pd.to_datetime(pdf["ds"])
series = pdf.set_index("ds")["y"].asfreq("D")
train, valid, test = series.iloc[:112], series.iloc[112:126], series.iloc[126:]
assert (len(train),len(valid),len(test)) == (112,14,14)
def errors(actual, prediction):
 a, p = np.asarray(actual), np.asarray(prediction)
 assert a.shape == p.shape and np.isfinite(p).all()
 return {"MAE": float(np.abs(a-p).mean()),
         "RMSE": float(np.sqrt(((a-p)**2).mean()))}


## Seasonal naive baseline


In [ ]:
def seasonal_naive(history, dates):
 # Repeat the last observed week for the entire forecast horizon.
 return pd.Series(np.resize(history.iloc[-7:].to_numpy(),len(dates)),index=dates)
baseline_pred = seasonal_naive(train, valid.index)
print("Validation seasonal naive:", errors(valid,baseline_pred))


## ARIMA


In [ ]:
def arima_forecast(history, dates):
 fitted = ARIMA(history,order=(2,1,2)).fit()
 forecast = fitted.get_forecast(steps=len(dates))
 pred = pd.Series(np.asarray(forecast.predicted_mean),index=dates)
 return pred, forecast.conf_int(alpha=0.05)
arima_pred, interval = arima_forecast(train,valid.index)
print("Validation ARIMA:",errors(valid,arima_pred))
print(interval.head())
# Inspect any convergence warnings. This fixed order is a teaching choice.


## Prophet


In [ ]:
def prophet_forecast(history, dates):
 frame = history.rename("y").rename_axis("ds").reset_index()
 model = Prophet(weekly_seasonality=True,
                 yearly_seasonality=False,daily_seasonality=False)
 model.fit(frame)
 forecast = model.predict(pd.DataFrame({"ds":dates}))
 return pd.Series(forecast.yhat.to_numpy(),index=dates)
prophet_pred = prophet_forecast(train,valid.index)
print("Validation Prophet:",errors(valid,prophet_pred))


## Holt Winters


In [ ]:
def holt_forecast(history, dates):
 model = ExponentialSmoothing(history,trend="add",seasonal="add",
                              seasonal_periods=7).fit()
 return pd.Series(np.asarray(model.forecast(len(dates))),index=dates)
holt_pred = holt_forecast(train,valid.index)
print("Validation Holt Winters:",errors(valid,holt_pred))


## Validation comparison and final test


In [ ]:
predictions = {"seasonal_naive":baseline_pred,"ARIMA":arima_pred,
               "Prophet":prophet_pred,"Holt_Winters":holt_pred}
scores = pd.DataFrame([{"model":name,**errors(valid,pred)}
                      for name,pred in predictions.items()])
scores = scores.sort_values(["MAE","model"]).reset_index(drop=True)
display(scores)
chosen = scores.iloc[0]["model"]
# Freeze the model family using validation. Test is still untouched.
fit_history = pd.concat([train,valid])
functions = {"seasonal_naive":seasonal_naive,
 "ARIMA":lambda history,dates: arima_forecast(history,dates)[0],
 "Prophet":prophet_forecast,"Holt_Winters":holt_forecast}
test_pred = functions[chosen](fit_history,test.index)
print("Frozen choice:",chosen,"Final test:",errors(test,test_pred))
result = pd.DataFrame({"ds":test.index,"actual":test.to_numpy(),
                      "prediction":test_pred.to_numpy()})
spark.createDataFrame(result).write.format("delta").mode("overwrite")\
 .saveAsTable(ns+".day18_forecast_test")
fig, ax = plt.subplots(figsize=(9,3))
test.plot(ax=ax,label="Actual")
test_pred.plot(ax=ax,label=chosen)
ax.set(ylabel="Units",xlabel="Date"); ax.legend()
display(fig); plt.close(fig)
